# NHL intro — sportsdataverse-py

Hockey from ESPN (`espn_nhl_*`) plus the NHL's own modern APIs: the game feed (`nhl_*`, e.g. `nhl_pbp`), player tracking (`nhl_edge_*`), aggregate stats (`nhl_stats_rest_*`), and the records site (`nhl_records_*`).

R companion: [fastRhockey](https://fastRhockey.sportsdataverse.org) (NHL + PWHL). Python neighbor: [nhl-api-py](https://github.com/coreyjs/nhl-api-py). Part of the [SportsDataverse](https://py.sportsdataverse.org/docs/ecosystem).

## Setup

```sh
pip install sportsdataverse
```

In [ ]:
import polars as pl
import sportsdataverse as sdv

## Teams

In [ ]:
teams = sdv.nhl.espn_nhl_teams()
teams.shape

In [ ]:
teams.select(['team_id', 'team_location', 'team_name', 'team_abbreviation']).head()

## Schedule (ESPN scoreboard)

In [ ]:
schedule = sdv.nhl.espn_nhl_schedule(dates=20240624)
schedule.select(['id', 'home_team_full_name', 'away_team_full_name', 'home_score', 'away_score']).head()

## Multi-season loader

In [ ]:
schedule_2024 = sdv.nhl.load_nhl_schedule(seasons=[2024])
schedule_2024.shape

## Play-by-play

In [ ]:
pbp = sdv.nhl.espn_nhl_pbp(game_id=401559700)
list(pbp.keys())[:8]

In [ ]:
plays = pl.from_pandas(pbp['plays'])
plays.select(['period_number', 'clock_display_value', 'text', 'scoring_play']).head()

## Shot-event filter

Filter the play-by-play to shot events — the simplest analysis primitive in hockey.

In [ ]:
shots = plays.filter(pl.col('text').str.contains('(?i)shot|goal'))
shots.shape

In [ ]:
(shots
    .group_by('type_text')
    .agg(pl.len().alias('events'))
    .sort('events', descending=True)
    .head(10))

## Pipeline example: goals per period

Filter to scoring plays, group by period.

In [ ]:
(plays
    .filter(pl.col('scoring_play') == True)
    .group_by('period_number')
    .agg(pl.len().alias('goals'))
    .sort('period_number'))

## Cross-references

- R companion: [fastRhockey](https://fastRhockey.sportsdataverse.org)
- Data source: ESPN NHL API + NHL stats API
- Python alternative: [nhl-api-py](https://github.com/coreyjs/nhl-api-py)
- Plotting: matplotlib, plotnine

## Where to go next

- API docs: `docs/docs/nhl/index.md`
- You're done with the intro series — head back to `01_quickstart.ipynb` for the cross-sport overview.